1단계: 환경 설정

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

2단계: 필요한 종속성 가져오기

In [2]:
import threading
import time
from datetime import timedelta

from mcp import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamable_http_client
from mcp.server import FastMCP
from strands import Agent
from strands.tools.mcp import MCPClient

STDIO MCP Client 생성

In [3]:
# Connect to an MCP server using stdio transport
stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

4단계 - 에이전트 구성 설정 및 호출

In [4]:
with stdio_mcp_client:
    # MCP 서버에서 도구 가져오기
    tools = stdio_mcp_client.list_tools_sync()

    # 이 도구들로 에이전트 생성
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent("Amazon Bedrock의 요금 모델은 무엇인가요? 간결하게 설명해 주세요.")

Amazon Bedrock의 요금 모델에 대해 검색해보겠습니다.
Tool #1: search_documentation

Tool #2: read_sections

Tool #3: read_documentation
완벽합니다! Amazon Bedrock의 요금 모델을 간결하게 설명드리겠습니다.

## Amazon Bedrock 요금 모델

Amazon Bedrock은 **API 사용량 기반의 유연한 요금제**를 제공합니다:

### 주요 요금 방식

1. **온디맨드 (On-Demand)**
   - 입력 및 출력 토큰 처리량에 따라 요금 부과
   - 사용한 만큼만 지불

2. **용량 단위 기반**
   - [임포트된 FM](https://docs.aws.amazon.com/bedrock/latest/userguide/import-pre-trained-model.html) 사용 시
   - 용량 단위(capacity units)로 측정하여 확장 가능

### 서비스 티어 (Service Tiers)

4가지 추론 서비스 티어를 통해 가용성, 지연시간, 비용을 유연하게 조정할 수 있습니다:

- **Reserved** (예약형): 별도 풀, SLA 보장, 최소 약정 필요
- **Priority** (우선순위): 온디맨드
- **Standard** (표준): 온디맨드  
- **Flex** (플렉스): 온디맨드

### 추가 비용 최적화 옵션

- **Provisioned Throughput**: 예측 가능한 워크로드에 적합
- **Batch Inference**: 대량 비동기 처리
- **Prompt Caching**: 프롬프트 재사용으로 비용 절감

요약하면, Bedrock은 **서버리스 종량제 방식**으로 인프라 관리 없이 사용한 토큰만큼만 비용을 지불하는 구조입니다.

### Streamable 부분은 생략하세요~

Streamable HTTP를 사용하여 MCP 서버에 연결

1단계: MCP 서버 생성

In [5]:
# Create an MCP server
mcp = FastMCP("Calculator Server")

/opt/conda/lib/python3.12/site-packages/pydantic_settings/sources/utils.py:51: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(  # noqa: B028  (called from many sources at differing depths, so no stacklevel is correct)


2단계: 도구 정의

In [6]:
@mcp.tool(description="Calculator tool which performs calculations")
def calculator(x: int, y: int) -> int:
    return x + y


@mcp.tool(description="This is a long running tool")
def long_running_tool(name: str) -> str:
    time.sleep(25)
    return f"Hello {name}"


def main():
    mcp.run(transport="streamable-http")

3 단계: streamable-http 서버로 스레드 시작

In [7]:
thread = threading.Thread(target=main)
thread.start()

INFO:     Started server process [28795]
INFO:     Waiting for application startup.


[10/31/26 18:58:23] INFO     StreamableHTTP session manager started                  ]8;id=8006567;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=8006568;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#131\131]8;;\

INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [8]:
def create_streamable_http_transport():
    return streamable_http_client("http://localhost:8000/mcp")


streamable_http_mcp_client = MCPClient(create_streamable_http_transport)

In [9]:
with streamable_http_mcp_client:
    tools = streamable_http_mcp_client.list_tools_sync()

    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = str(agent("What is 2 + 2?"))

[10/31/26 18:58:31] INFO     Created new transport with session ID:                  ]8;id=8006574;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=8006575;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             6dac9b80ff314681a2d8f862068865be                                                      

INFO:     127.0.0.1:52178 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=8006582;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8006583;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 6dac9b80ff314681a2d8f862068865be           ]8;id=8006590;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006591;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=8006597;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006598;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

INFO:     127.0.0.1:52190 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:52182 - "POST /mcp HTTP/1.1" 202 Accepted


... (8 outputs left out)

2 + 2 = **4**

[10/31/26 18:58:35] INFO     Terminating session: 6dac9b80ff314681a2d8f862068865be           ]8;id=8006643;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=8006644;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py#788\788]8;;\

INFO:     127.0.0.1:52238 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=8006649;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8006650;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

In [10]:
query = {"x": 10, "y": 20}

with streamable_http_mcp_client:
    # direct tool invocation
    result = streamable_http_mcp_client.call_tool_sync(
        tool_use_id="tool-123", name="calculator", arguments=query
    )

    # Process the result
    print(f"Calculation result: {result['content'][0]['text']}")

[10/31/26 18:58:38] INFO     Created new transport with session ID:                  ]8;id=8006655;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=8006656;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             005d788b1ecf45e68bbb851164bc1b60                                                      

INFO:     127.0.0.1:40906 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=8006661;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8006662;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 005d788b1ecf45e68bbb851164bc1b60           ]8;id=8006667;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006668;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=8006673;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006674;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

INFO:     127.0.0.1:40922 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:40918 - "POST /mcp HTTP/1.1" 202 Accepted


... (9 outputs left out)

                    INFO     Terminating session: 005d788b1ecf45e68bbb851164bc1b60           ]8;id=8006715;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=8006716;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py#788\788]8;;\

INFO:     127.0.0.1:40944 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=8006721;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8006722;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     GET stream disconnected, reconnecting in 1000ms...              ]8;id=8006728;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006729;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#298\298]8;;\

In [11]:
with streamable_http_mcp_client:
    try:
        result = streamable_http_mcp_client.call_tool_sync(
            tool_use_id="tool-123",
            name="long_running_tool",
            arguments={"name": "Amazon"},
            read_timeout_seconds=timedelta(seconds=30),
        )

        if result["status"] == "error":
            print(f"Tool execution failed: {result['content'][0]['text']}")
        else:
            print(f"Tool execution succeeded: {result['content'][0]['text']}")
    except Exception as e:
        print(f"Tool call timed out or failed: {str(e)}")

[10/31/26 18:59:06] INFO     Created new transport with session ID:                  ]8;id=8006734;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=8006735;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#281\281]8;;\
                             4e867242b38e4780a71640b28da9c3db                                                      

INFO:     127.0.0.1:58196 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=8006740;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8006741;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 4e867242b38e4780a71640b28da9c3db           ]8;id=8006746;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006747;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=8006752;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=8006753;file:///opt/conda/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

INFO:     127.0.0.1:58204 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58208 - "POST /mcp HTTP/1.1" 202 Accepted


... (8 outputs left out)

Tool execution succeeded: Hello Amazon


                    INFO     Terminating session: 4e867242b38e4780a71640b28da9c3db           ]8;id=8006794;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=8006795;file:///opt/conda/lib/python3.12/site-packages/mcp/server/streamable_http.py#788\788]8;;\

INFO:     127.0.0.1:43292 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=8006800;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8006801;file:///opt/conda/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

여러 MCP 서버와 상호 작용하기

1단계: stdio 전송을 사용하여 MCP 서버(AWS Documentation MCP Server)에 연결

In [12]:
aws_docs_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

2단계: stdio 전송을 사용하여 MCP 서버(AWS CDK MCP Server)에 연결

In [19]:
iac_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command="uvx", args=["awslabs.aws-iac-mcp-server@latest"])
    )
)

3단계: MCP 서버를 사용하여 에이전트 생성

In [20]:
with aws_docs_mcp_client, iac_mcp_client:
    # MCP 서버에서 도구 가져오기
    tools = aws_docs_mcp_client.list_tools_sync() + iac_mcp_client.list_tools_sync()

    # 이러한 도구로 에이전트 생성
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools,
        #max_parallel_tools=2
    )

    response = agent(
        "Amazon Bedrock의 요금 모델은 어떻게 되나요? 간략하게 설명해 주세요. 또한, CDK와 관련하여 권장되는 모범 사례(best practices)는 무엇인가요?"
    )

Amazon Bedrock의 요금 모델과 CDK 모범 사례에 대해 알아보겠습니다.
Tool #1: search_documentation

Tool #2: cdk_best_practices
이제 Bedrock 요금에 대한 더 자세한 정보를 확인하겠습니다.
Tool #3: read_sections

Tool #4: read_documentation
완벽합니다! 이제 두 질문에 대한 답변을 제공하겠습니다.

## Amazon Bedrock 요금 모델 📊

Amazon Bedrock의 요금 모델은 다음과 같습니다:

### 1. **유연한 API 소비 기반 가격 정책**
- **온디맨드(On-Demand) 사용**: 입력 및 출력 토큰 처리량에 대해 사용한 만큼 지불
- **임포트된 FM 사용**: 용량 단위(capacity units) 기반으로 측정 및 과금

### 2. **4가지 서비스 티어 (Service Tiers)**
모델이나 API 변경 없이 가용성, 지연시간, 비용을 조정할 수 있습니다:

- **Reserved (예약형)**: 별도 SLA와 최소 약정이 있는 독립 풀
- **Priority (우선순위)**: 온디맨드, 모델당 할당량 공유
- **Standard (표준)**: 온디맨드, 모델당 할당량 공유
- **Flex (유연형)**: 온디맨드, 모델당 할당량 공유

### 3. **추가 용량 제어 옵션**
- **Provisioned Throughput**: 사용자 정의 및 임포트된 모델용 예약 처리량
- **Batch Inference**: 일괄 처리 추론
- **Cross-Region Inference**: 리전 간 추론

### 4. **비용 최적화 기능**
- **Intelligent Prompt Routing**: 품질 요구사항을 충족하는 가장 비용 효율적인 모델로 자동 라우팅
- **Prompt Caching**: 캐시된 프롬프트 접두사 재사용으로 비용 절감
- **실시간 및 배치 처리**: 지연시간과 비용 요구사항에 맞는 처리 모드 선택